<a href="https://colab.research.google.com/github/JPAmewu/My_Capstone_1_Imperial/blob/main/Week_11/02_Notebook/Week_11_Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 11 Capstone — Bayesian Optimisation

This notebook appends the confirmed Week 10 observations to the Week 10 dataset, creating the Week 11 dataset before EDA and GP/UCB query selection.


For each function for Week_10, I will follow this sequence:
1. Create week_11_dataset
2. Extract function
3. Check shapes and missing values
4. Calculate summary statistics
5. Plot the output trend
6. Plot the best-so-far trend
7. Identify the best query point
8. Plot the correlation heatmap
9. Plot each input against the output
10. Fit a Gaussian Process surrogate
11. Apply UCB to generate the next query point

In [ ]:
# 1. Unzip the Week 10 dataset

import os
import shutil
import zipfile

import numpy as np

zip_path = "/content/week_10_dataset.zip"
extract_path = "/content/week_10_dataset"

# Remove an old extracted copy, if present.
if os.path.exists(extract_path):
    shutil.rmtree(extract_path)

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Week 10 dataset extracted successfully.")


In [ ]:
# 2. Check the extracted .npy files

for root, folders, files in os.walk(extract_path):
    for file in files:
        if file.endswith(".npy"):
            print(os.path.join(root, file))

In [ ]:
# 3. Load and validate the Week 10 dataset

EXPECTED_DIMENSIONS = {
    1: 2,
    2: 2,
    3: 3,
    4: 4,
    5: 4,
    6: 5,
    7: 6,
    8: 8,
}


def find_npy_file(folder, function_number, data_type):
    """Return the unique input or output array for a function."""
    if data_type not in {"input", "output"}:
        raise ValueError("data_type must be 'input' or 'output'.")

    expected_name = f"function_{function_number}_{data_type}s.npy"
    matching_files = sorted(
        os.path.join(root, filename)
        for root, _, files in os.walk(folder)
        for filename in files
        if filename.lower() == expected_name
    )

    if not matching_files:
        raise FileNotFoundError(
            f"No exact {data_type} file found for Function "
            f"{function_number}: {expected_name}"
        )

    if len(matching_files) > 1:
        raise RuntimeError(
            f"Multiple {data_type} files found for Function "
            f"{function_number}: {matching_files}"
        )

    return matching_files[0]


week_10_X = {}
week_10_y = {}

for function_number in range(1, 9):
    input_file = find_npy_file(
        extract_path,
        function_number,
        "input",
    )
    output_file = find_npy_file(
        extract_path,
        function_number,
        "output",
    )

    expected_dimension = EXPECTED_DIMENSIONS[function_number]
    inputs = np.asarray(np.load(input_file), dtype=float)
    outputs = np.asarray(np.load(output_file), dtype=float).reshape(-1, 1)

    if inputs.ndim == 1:
        if inputs.size % expected_dimension:
            raise ValueError(
                f"Function {function_number}: {inputs.size} input values "
                f"cannot be reshaped to dimension {expected_dimension}."
            )
        inputs = inputs.reshape(-1, expected_dimension)

    if inputs.ndim != 2 or inputs.shape[1] != expected_dimension:
        raise ValueError(
            f"Function {function_number}: expected a two-dimensional input "
            f"array with {expected_dimension} columns, got {inputs.shape}."
        )

    if not np.isfinite(inputs).all() or not np.isfinite(outputs).all():
        raise ValueError(
            f"Function {function_number}: input or output contains "
            "non-finite values."
        )

    week_10_X[function_number] = inputs
    week_10_y[function_number] = outputs

    print(f"Function {function_number}")
    print("Input shape :", inputs.shape)
    print("Output shape:", outputs.shape)
    print("-" * 40)


In [ ]:
# Validate input-output alignment without modifying the raw dataset.

for function_number in range(1, 9):
    input_rows = week_10_X[function_number].shape[0]
    output_rows = week_10_y[function_number].shape[0]

    if input_rows != output_rows:
        raise ValueError(
            f"Function {function_number}: found {input_rows} input rows and "
            f"{output_rows} outputs. Resolve the mismatch against the original "
            "query history before creating the Week 11 dataset."
        )

print("All Week 10 input and output arrays are aligned.")


In [ ]:
# Raw Week 10 files remain unchanged.
print("Validation complete; no source dataset files were overwritten.")


In [ ]:
# Confirmed Week 10 query points and returned outputs.
new_X = {key: np.array(value, dtype=float) for key, value in {1: [[0.379403, 0.071186]], 2: [[0.329121, 0.99795]], 3: [[0.447317, 0.65893, 0.324854]], 4: [[0.587928, 0.514452, 0.045793, 0.032616]], 5: [[0.005552, 0.918769, 0.692571, 0.988052]], 6: [[0.853051, 0.152023, 0.700695, 0.554516, 0.611033]], 7: [[0.107447, 0.625529, 0.309529, 0.130105, 0.44366, 0.667717]], 8: [[0.546897, 0.163123, 0.136484, 0.078398, 0.771803, 0.070794, 0.119397, 0.111488]]}.items()}
new_y = {key: np.array(value, dtype=float) for key, value in {1: [[-3.089423814911752e-96]], 2: [[0.049406406222616564]], 3: [[-0.08189344986506433]], 4: [[-23.42280313862202]], 5: [[430.8031249775375]], 6: [[-1.1717131510084098]], 7: [[1.009894033457839]], 8: [[9.6998918101966]]}.items()}

for function_number in range(1, 9):
    print(
        f"Function {function_number}: "
        f"input {new_X[function_number].shape}, "
        f"output {new_y[function_number].shape}"
    )


In [ ]:
week_11_X = {}
week_11_y = {}

for function_number in range(1, 9):
    old_X = week_10_X[function_number]
    old_y = week_10_y[function_number]
    current_X = new_X[function_number]
    current_y = new_y[function_number]

    if old_X.shape[0] != old_y.shape[0]:
        raise ValueError(
            f"Function {function_number}: existing input/output row mismatch."
        )

    if old_X.shape[1] != current_X.shape[1]:
        raise ValueError(
            f"Function {function_number}: input dimension mismatch. "
            f"Existing data has {old_X.shape[1]} columns, "
            f"but the new input has {current_X.shape[1]} columns."
        )

    if current_X.shape[0] != current_y.shape[0]:
        raise ValueError(
            f"Function {function_number}: new input/output row mismatch."
        )

    week_11_X[function_number] = np.vstack([old_X, current_X])
    week_11_y[function_number] = np.vstack([old_y, current_y])

    print(f"Function {function_number}")
    print("Week 10 input shape :", old_X.shape)
    print("Week 10 output shape:", old_y.shape)
    print("Week 11 input shape:", week_11_X[function_number].shape)
    print("Week 11 output shape:", week_11_y[function_number].shape)
    print("-" * 45)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

week_11_folder = "/content/drive/MyDrive/week_11_dataset"
os.makedirs(week_11_folder, exist_ok=True)

for function_number in range(1, 9):
    input_file = os.path.join(
        week_11_folder,
        f"function_{function_number}_inputs.npy",
    )
    output_file = os.path.join(
        week_11_folder,
        f"function_{function_number}_outputs.npy",
    )

    np.save(input_file, week_11_X[function_number])
    np.save(output_file, week_11_y[function_number])

print("Week 11 dataset saved successfully.")
print("Folder:", week_11_folder)


In [ ]:
for function_number in range(1, 9):

    saved_X = np.load(
        os.path.join(
            week_11_folder,
            f"function_{function_number}_inputs.npy"
        )
    )

    saved_y = np.load(
        os.path.join(
            week_11_folder,
            f"function_{function_number}_outputs.npy"
        )
    )

    status = (
        "MATCHED"
        if saved_X.shape[0] == saved_y.shape[0]
        else "NOT MATCHED"
    )

    print(
        f"Function {function_number}: "
        f"input {saved_X.shape}, "
        f"output {saved_y.shape} — {status}"
    )

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")




**Function**-by-function EDA function

1. Loading of inputs and outputs
2. Shape checks
3. Missing and infinite-value checks
4. Summary statistics
5. Output trend
6. Best-so-far trend
7. Best observed query point
8. Correlation heatmap
9. Input-output scatter plots
10. Top five observations

In [ ]:
def conduct_function_eda(function_number, dataset_folder):
    """
    Conduct exploratory data analysis for one black-box function.

    Parameters
    ----------
    function_number : int
        Function number from 1 to 8.

    dataset_folder : str
        Folder containing the .npy files.
    """

    # ---------------------------------------------------------
    # 1. Build the file paths
    # ---------------------------------------------------------
    input_path = os.path.join(
        dataset_folder,
        f"function_{function_number}_inputs.npy"
    )

    output_path = os.path.join(
        dataset_folder,
        f"function_{function_number}_outputs.npy"
    )

    # ---------------------------------------------------------
    # 2. Check that the files exist
    # ---------------------------------------------------------
    if not os.path.exists(input_path):
        raise FileNotFoundError(
            f"Input file not found: {input_path}"
        )

    if not os.path.exists(output_path):
        raise FileNotFoundError(
            f"Output file not found: {output_path}"
        )

    # ---------------------------------------------------------
    # 3. Load the arrays
    # ---------------------------------------------------------
    X = np.load(input_path)
    y = np.load(output_path)

    # Make sure X is two-dimensional
    if X.ndim == 1:
        X = X.reshape(-1, 1)

    # Convert output from shape (n, 1) into shape (n,)
    y = y.reshape(-1)

    # ---------------------------------------------------------
    # 4. Check input-output matching
    # ---------------------------------------------------------
    if X.shape[0] != y.shape[0]:
        raise ValueError(
            f"Function {function_number} is not matched: "
            f"{X.shape[0]} input rows and {y.shape[0]} outputs."
        )

    # ---------------------------------------------------------
    # 5. Create a DataFrame
    # ---------------------------------------------------------
    input_columns = [
        f"x{i + 1}" for i in range(X.shape[1])
    ]

    df = pd.DataFrame(
        X,
        columns=input_columns
    )

    df["output"] = y

    # ---------------------------------------------------------
    # 6. Basic information
    # ---------------------------------------------------------
    print("\n")
    print("=" * 70)
    print(f"FUNCTION {function_number} EDA")
    print("=" * 70)

    print("\nDataset structure")
    print("-" * 40)
    print("Input shape :", X.shape)
    print("Output shape:", y.shape)
    print("Observations:", X.shape[0])
    print("Dimensions  :", X.shape[1])

    print("\nFirst five observations")
    print("-" * 40)
    display(df.head())

    # ---------------------------------------------------------
    # 7. Data-quality checks
    # ---------------------------------------------------------
    missing_values = df.isnull().sum()
    infinite_values = pd.Series(
        np.isinf(df.to_numpy()).sum(axis=0),
        index=df.columns
    )

    duplicate_rows = df.duplicated().sum()

    print("\nMissing values")
    print("-" * 40)
    print(missing_values)

    print("\nInfinite values")
    print("-" * 40)
    print(infinite_values)

    print("\nDuplicate rows")
    print("-" * 40)
    print(duplicate_rows)

    # ---------------------------------------------------------
    # 8. Summary statistics
    # ---------------------------------------------------------
    summary = df.describe().T

    summary["median"] = df.median()
    summary["variance"] = df.var()
    summary["skewness"] = df.skew()

    print("\nSummary statistics")
    print("-" * 40)
    display(summary)

    # ---------------------------------------------------------
    # 9. Find the best observed result
    # Maximisation objective
    # ---------------------------------------------------------
    best_index = int(np.argmax(y))
    best_output = float(y[best_index])
    best_input = X[best_index]

    print("\nBest observed result")
    print("-" * 40)
    print("Best array index       :", best_index)
    print("Best observation number:", best_index + 1)
    print("Best query point       :", best_input)
    print("Best output value      :", best_output)

    # ---------------------------------------------------------
    # 10. Top five observations
    # ---------------------------------------------------------
    top_five = (
        df.sort_values(
            by="output",
            ascending=False
        )
        .head(5)
    )

    print("\nTop five observations")
    print("-" * 40)
    display(top_five)

    # ---------------------------------------------------------
    # 11. Output trend
    # ---------------------------------------------------------
    observation_numbers = np.arange(1, len(y) + 1)

    plt.figure(figsize=(11, 5))

    plt.plot(
        observation_numbers,
        y,
        marker="o"
    )

    plt.scatter(
        best_index + 1,
        best_output,
        marker="*",
        s=250,
        label="Best observed output"
    )

    plt.xlabel("Observation number")
    plt.ylabel("Output")
    plt.title(
        f"Function {function_number}: Output Trend"
    )

    plt.legend()
    plt.tight_layout()
    plt.show()

    # ---------------------------------------------------------
    # 12. Best-so-far trend
    # ---------------------------------------------------------
    best_so_far = np.maximum.accumulate(y)

    plt.figure(figsize=(11, 5))

    plt.plot(
        observation_numbers,
        best_so_far,
        marker="o"
    )

    plt.xlabel("Observation number")
    plt.ylabel("Best output found so far")
    plt.title(
        f"Function {function_number}: Best-So-Far Trend"
    )

    plt.tight_layout()
    plt.show()

    # ---------------------------------------------------------
    # 13. Output distribution
    # ---------------------------------------------------------
    plt.figure(figsize=(9, 5))

    plt.hist(
        y,
        bins= 8,
        edgecolor="black"
    )

    plt.axvline(
        np.mean(y),
        linestyle="--",
        label=f"Mean = {np.mean(y):.5g}"
    )

    plt.axvline(
        np.median(y),
        linestyle=":",
        label=f"Median = {np.median(y):.5g}"
    )

    plt.xlabel("Output")
    plt.ylabel("Frequency")
    plt.title(
        f"Function {function_number}: Output Distribution"
    )

    plt.legend()
    plt.tight_layout()
    plt.show()

    # ---------------------------------------------------------
    # 14. Correlation heatmap
    # ---------------------------------------------------------
    correlation = df.corr(numeric_only=True)

    plt.figure(
        figsize=(
            max(7, X.shape[1] + 3),
            max(5, X.shape[1] + 2)
        )
    )

    sns.heatmap(
        correlation,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0
    )

    plt.title(
        f"Function {function_number}: Correlation Heatmap"
    )

    plt.tight_layout()
    plt.show()

    # ---------------------------------------------------------
    # 15. Each input against the output
    # ---------------------------------------------------------
    for column in input_columns:

        plt.figure(figsize=(8, 5))

        plt.scatter(
            df[column],
            df["output"],
            alpha=0.75
        )

        plt.scatter(
            df.loc[best_index, column],
            df.loc[best_index, "output"],
            marker="*",
            s=250,
            label="Best observed point"
        )

        plt.xlabel(column)
        plt.ylabel("Output")

        plt.title(
            f"Function {function_number}: "
            f"{column} Against Output"
        )

        plt.legend()
        plt.tight_layout()
        plt.show()

    # ---------------------------------------------------------
    # 16. Input-output correlations
    # ---------------------------------------------------------
    output_correlations = (
        correlation["output"]
        .drop("output")
        .sort_values(
            key=lambda values: values.abs(),
            ascending=False
        )
    )

    print("\nInput-output correlations")
    print("-" * 40)
    print(output_correlations)

    # ---------------------------------------------------------
    # 17. Concise report
    # ---------------------------------------------------------
    print("\nConcise EDA report")
    print("-" * 40)

    print(f"Observations       : {X.shape[0]}")
    print(f"Input dimensions   : {X.shape[1]}")
    print(f"Missing values     : {df.isnull().sum().sum()}")
    print(f"Infinite values    : {np.isinf(df.to_numpy()).sum()}")
    print(f"Duplicate rows     : {duplicate_rows}")
    print(f"Minimum output     : {np.min(y):.10g}")
    print(f"Mean output        : {np.mean(y):.10g}")
    print(f"Median output      : {np.median(y):.10g}")
    print(f"Maximum output     : {np.max(y):.10g}")
    print(f"Best observation   : {best_index + 1}")
    print(f"Best query point   : {best_input}")
    print(f"Best output        : {best_output:.10g}")

    # Return the important objects
    return {
        "function": function_number,
        "X": X,
        "y": y,
        "dataframe": df,
        "summary": summary,
        "correlation": correlation,
        "best_index": best_index,
        "best_input": best_input,
        "best_output": best_output,
        "best_so_far": best_so_far,
        "top_five": top_five
    }

In [ ]:
# Conduct Function 1 EDA

eda_1 = conduct_function_eda(
    function_number=1,
    dataset_folder=week_11_folder
)

In [ ]:
# Conduct Function 2 EDA
eda_2 = conduct_function_eda(
    function_number=2,
    dataset_folder=week_11_folder
)

In [ ]:
# Conduct Function 3 EDA
eda_3 = conduct_function_eda(
    function_number=3,
    dataset_folder=week_11_folder
)

In [ ]:
# Conduct Function 4 EDA
eda_4 = conduct_function_eda(
    function_number=4,
    dataset_folder=week_11_folder
)

In [ ]:
# Conduct Function 5 EDA
eda_5 = conduct_function_eda(
    function_number=5,
    dataset_folder=week_11_folder
)

In [ ]:
# Conduct Function 6 EDA
eda_6 = conduct_function_eda(
    function_number=6,
    dataset_folder=week_11_folder
)

In [ ]:
# Conduct Function 7 EDA
eda_7 = conduct_function_eda(
    function_number=7,
    dataset_folder=week_11_folder
)

In [ ]:
# Conduct Function 8 EDA
eda_8 = conduct_function_eda(
    function_number=8,
    dataset_folder=week_11_folder
)

In [ ]:
all_eda_results = [
    eda_1,
    eda_2,
    eda_3,
    eda_4,
    eda_5,
    eda_6,
    eda_7,
    eda_8
]

combined_summary = []

for result in all_eda_results:

    X = result["X"]
    y = result["y"]

    combined_summary.append({
        "Function": result["function"],
        "Observations": X.shape[0],
        "Dimensions": X.shape[1],
        "Minimum_Output": np.min(y),
        "Mean_Output": np.mean(y),
        "Median_Output": np.median(y),
        "Maximum_Output": np.max(y),
        "Best_Observation": result["best_index"] + 1,
        "Best_Query_Point": result["best_input"],
        "Best_Output": result["best_output"]
    })

combined_summary_df = pd.DataFrame(combined_summary)

display(combined_summary_df)

In [ ]:
# Import the required Gaussian Process classes.
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel,
)


Define the UCB acquisition function

For maximisation, UCB is:

 UCB(x)=mu(x)+ kappasigma(x)

where:

* mu(x) is the GP-predicted mean;
* sigma(x) is the GP uncertainty;
* κappa controls exploration;
* a larger κappa produces more exploration;
* a smaller κappa produces more exploitation.

In [ ]:
def upper_confidence_bound(mean, std, kappa=2.0):
    """
    UCB acquisition function for maximisation.
    """
    return mean + (kappa * std)

In [ ]:
def generate_bo_query(
    function_number,
    dataset_folder,
    kappa=2.0,
    number_of_candidates=100000,
    random_state=42,
):
    """Fit a GP and select a new, rounded query with UCB."""
    if function_number not in EXPECTED_DIMENSIONS:
        raise ValueError("function_number must be between 1 and 8.")
    if kappa < 0:
        raise ValueError("kappa must be non-negative.")
    if number_of_candidates < 1:
        raise ValueError("number_of_candidates must be positive.")

    input_path = os.path.join(
        dataset_folder,
        f"function_{function_number}_inputs.npy",
    )
    output_path = os.path.join(
        dataset_folder,
        f"function_{function_number}_outputs.npy",
    )

    X = np.asarray(np.load(input_path), dtype=float)
    y = np.asarray(np.load(output_path), dtype=float).reshape(-1)
    expected_dimension = EXPECTED_DIMENSIONS[function_number]

    if X.ndim == 1:
        if X.size % expected_dimension:
            raise ValueError(
                f"Function {function_number}: cannot reshape {X.size} input "
                f"values to dimension {expected_dimension}."
            )
        X = X.reshape(-1, expected_dimension)

    if X.ndim != 2 or X.shape[1] != expected_dimension:
        raise ValueError(
            f"Function {function_number}: expected input shape "
            f"(n, {expected_dimension}), got {X.shape}."
        )

    if X.shape[0] != y.shape[0]:
        raise ValueError(
            f"Function {function_number}: {X.shape[0]} input rows but "
            f"{y.shape[0]} outputs."
        )

    if not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError(
            f"Function {function_number} contains non-finite values."
        )

    if np.any((X < 0.0) | (X > 1.0)):
        raise ValueError(
            f"Function {function_number}: inputs must lie in [0, 1]."
        )

    y_mean = np.mean(y)
    y_std = np.std(y)
    y_scaled = y - y_mean if np.isclose(y_std, 0.0) else (y - y_mean) / y_std

    dimensions = X.shape[1]
    kernel = (
        ConstantKernel(
            constant_value=1.0,
            constant_value_bounds=(1e-3, 1e3),
        )
        * Matern(
            length_scale=np.full(dimensions, 0.2),
            length_scale_bounds=(0.01, 2.0),
            nu=2.5,
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-2),
        )
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=False,
        n_restarts_optimizer=10,
        random_state=random_state,
    )
    gp.fit(X, y_scaled)

    rng = np.random.default_rng(random_state + function_number)
    candidates = rng.uniform(
        low=0.0,
        high=1.0,
        size=(number_of_candidates, dimensions),
    )

    # Exclude points that duplicate an observation after submission rounding.
    observed_points = {tuple(row) for row in np.round(X, 6)}
    rounded_candidates = np.round(candidates, 6)
    is_new = np.array(
        [tuple(row) not in observed_points for row in rounded_candidates],
        dtype=bool,
    )
    candidates = candidates[is_new]
    rounded_candidates = rounded_candidates[is_new]

    if not len(candidates):
        raise RuntimeError("No unevaluated candidate points were generated.")

    predicted_mean, predicted_std = gp.predict(candidates, return_std=True)
    ucb = upper_confidence_bound(predicted_mean, predicted_std, kappa=kappa)

    best_candidate_index = int(np.argmax(ucb))
    next_query_rounded = rounded_candidates[best_candidate_index]
    submission_query = "-".join(
        f"{value:.6f}" for value in next_query_rounded
    )

    return {
        "function": function_number,
        "query_array": next_query_rounded,
        "submission_query": submission_query,
        "predicted_mean_scaled": predicted_mean[best_candidate_index],
        "predicted_std_scaled": predicted_std[best_candidate_index],
        "ucb_value": ucb[best_candidate_index],
        "fitted_kernel": gp.kernel_,
    }


In [ ]:
# Function 1 Query
week_11_folder = "/content/drive/MyDrive/week_11_dataset"

query_1 = generate_bo_query(
    function_number=1,
    dataset_folder=week_11_folder,
    kappa=2.0,
    number_of_candidates=100000,
    random_state=42
)

print("Function 1 query:")
print(query_1["submission_query"])

In [ ]:
all_queries = {}

for function_number in range(1, 9):

    result = generate_bo_query(
        function_number=function_number,
        dataset_folder=week_11_folder,
        kappa=2.0,
        number_of_candidates=100000,
        random_state=42
    )

    all_queries[function_number] = result

    print(
        f"Function {function_number}: "
        f"{result['submission_query']}"
    )

In [ ]:
# Print only the submission strings
print("WEEK 12 QUERY POINTS")
print("=" * 50)

for function_number in range(1, 9):
    print(all_queries[function_number]["submission_query"])

In [ ]:
submission_file = os.path.join(
    week_11_folder,
    "week_12_query_points.txt",
)

with open(submission_file, "w", encoding="utf-8") as file:
    for function_number in range(1, 9):
        query_string = all_queries[function_number]["submission_query"]
        file.write(f"Function_{function_number}:{query_string}\n")

print("Saved to:")
print(submission_file)
